# LLM Distillation / Fine-Tuning Demo

This notebook walks through the full pipeline for fine-tuning a small LLM on a
**single-file HTML website generation** task using **Unsloth** + **HuggingFace TRL**.

The workflow is:
1. **Generate training data** — query a powerful model via OpenRouter to get high-quality HTML examples.
2. **Evaluate the base model** — run the small model before fine-tuning on the eval test cases.
3. **Fine-tune** — apply QLoRA with Unsloth.
4. **Evaluate the fine-tuned model** — compare before / after quality in the browser.

Each pipeline step is driven by a **function imported directly from the corresponding
script file**, so all output appears live inside the cell rather than in a subprocess.
Training prompts and evaluation test cases are stored in **`prompts.json`**.

---
**Hardware:** RTX 2080 Ti (22 GB VRAM) — the defaults are tuned for this card.  
**Environment:** run this notebook inside the `distill_dev` Docker container started with:
```bash
sudo docker compose -f docker-compose-distill.yml up
```
Then open `http://localhost:8888`.

## 0. Configuration

In [2]:
import os, sys, json

# Make the distillation scripts importable when the notebook is opened from
# any working directory (e.g. the repo root inside the Docker container).
SCRIPT_DIR = os.path.dirname(os.path.abspath("distillation_demo.ipynb"))
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)

# -----------------------------------------------------------------------
# Set your keys here OR export them as environment variables before
# starting the container.
# -----------------------------------------------------------------------
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "<your-openrouter-key>")
HF_TOKEN          = os.getenv("HF_TOKEN", "")   # optional — only needed to push to Hub

# Local llama.cpp-compatible server URL (when using provider="local").
# Defaults to http://localhost:8080 but you can set LLAMA_CPP_SERVER_URL in your env.
LLAMA_CPP_SERVER_URL = os.getenv("LLAMA_CPP_SERVER_URL", "http://host.docker.internal:1337")

# Teacher model used to generate training data (via OpenRouter or local server)
# When using provider="local", this should match a model name served by your local server.
TEACHER_MODEL = "deepseek-ai/DeepSeek-R1-0528:together"
# Alternatively: "mistralai/mistral-large", "openai/gpt-4o-mini", etc.

# Small student model to fine-tune
STUDENT_MODEL = "ibm-granite/granite-4.0-h-1b"
# Alternatives: "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
#               "ibm-granite/granite-3.1-2b-instruct"

# Prompts file — contains training topics, instruction templates, and eval test cases
PROMPTS_FILE = os.path.join(SCRIPT_DIR, "prompts.json")

# Paths
DATASET_PATH   = os.path.join(SCRIPT_DIR, "website_dataset.jsonl")
ADAPTER_DIR    = os.path.join(SCRIPT_DIR, "outputs", "lora_model")
EVAL_BASE_DIR  = os.path.join(SCRIPT_DIR, "eval_results", "base")
EVAL_FT_DIR    = os.path.join(SCRIPT_DIR, "eval_results", "finetuned")

GGUF_DIR       = os.path.join(SCRIPT_DIR, "outputs", "gguf_model")
# GGUF quantization: q4_k_m (default), q5_k_m, q8_0, f16, q4_0, q5_0, q2_k, q3_k_m
GGUF_QUANT     = "q4_k_m"

N_SAMPLES      = 200   # number of training examples to generate
EPOCHS         = 1
BATCH_SIZE     = 2

GRAD_ACCUM     = 4     # effective batch = 2*4 = 8

print("Configuration loaded.")
print(f"  Script directory : {SCRIPT_DIR}")
print(f"  Prompts file     : {PROMPTS_FILE}")
print(f"  LLAMA_CPP_SERVER_URL : {LLAMA_CPP_SERVER_URL}")

Configuration loaded.
  Script directory : /code
  Prompts file     : /code/prompts.json
  LLAMA_CPP_SERVER_URL : http://host.docker.internal:1337


## 0b. Inspect Prompts File

All training topics, instruction templates, and evaluation test cases live in
`prompts.json`. Edit that file to add new topics or change eval cases without
touching any Python script.

In [3]:
with open(PROMPTS_FILE) as fh:
    prompts = json.load(fh)

print(f"Training topics      : {len(prompts['training_topics'])}")
print(f"Instruction templates: {len(prompts['instruction_templates'])}")
print(f"Eval test cases      : {len(prompts['eval_test_cases'])}")

print("\n--- First 5 training topics ---")
for t in prompts['training_topics'][:5]:
    print(f"  · {t}")

print("\n--- Eval test case IDs ---")
for tc in prompts['eval_test_cases']:
    print(f"  · {tc['id']}: {tc['instruction'][:80]}...")

Training topics      : 80
Instruction templates: 8
Eval test cases      : 8

--- First 5 training topics ---
  · a simple todo list app with add, complete, and delete functionality
  · a Pomodoro productivity timer with configurable work and break cycles
  · a kanban board with three columns: To Do, In Progress, and Done
  · a daily schedule planner with hourly time slots from 6 AM to midnight
  · a goal tracking page with labelled progress bars and percentage complete

--- Eval test case IDs ---
  · weather_chart: Build a self-contained HTML page for a 7-day weather forecast with SVG bar chart...
  · pixel_art_editor: Create a single-file HTML website: a pixel art editor with a 16x16 drawing grid,...
  · chat_ui: Write a complete single-file HTML + CSS + JS website that implements a chat appl...
  · spreadsheet_grid: Generate a responsive single-file HTML website for a mini spreadsheet with edita...
  · saas_landing: Produce a polished, self-contained HTML file that acts as an animate

## 1. Generate Training Data

We call a capable teacher model via **OpenRouter** to produce high-quality
single-file HTML examples across a variety of website types.  
Each record is saved as a JSON line: `{instruction, input, output}`.  
Topics and templates are read from `prompts.json`.

The cell imports `generate_dataset` directly from `generate_training_data.py`
so all progress output appears live in the cell.

In [4]:
from generate_training_data import generate_dataset

generate_dataset(
    api_key=HF_TOKEN,
    model=TEACHER_MODEL,
    n_samples=N_SAMPLES,
    output=DATASET_PATH,
    prompts_file=PROMPTS_FILE,
    provider="local",
    local_server_url=LLAMA_CPP_SERVER_URL,
    store_mode="file_path",
    html_output_dir=os.path.join(SCRIPT_DIR, "html_outputs")
)

Loaded 80 topics and 8 templates from /code/prompts.json.
Resuming — 3 samples already in /code/website_dataset.jsonl.


Generating:   0%|          | 0/200 [00:00<?, ?it/s]

  [attempt 1/6] local server error: HTTPConnectionPool(host='host.docker.internal', port=1337): Max retries exceeded with url: /v1/chat/completions (Caused by NameResolutionError("HTTPConnection(host='host.docker.internal', port=1337): Failed to resolve 'host.docker.internal' ([Errno -2] Name or service not known)")). Retrying in 2.7s ...
  [attempt 2/6] local server error: HTTPConnectionPool(host='host.docker.internal', port=1337): Max retries exceeded with url: /v1/chat/completions (Caused by NameResolutionError("HTTPConnection(host='host.docker.internal', port=1337): Failed to resolve 'host.docker.internal' ([Errno -2] Name or service not known)")). Retrying in 4.5s ...
  [attempt 3/6] local server error: HTTPConnectionPool(host='host.docker.internal', port=1337): Max retries exceeded with url: /v1/chat/completions (Caused by NameResolutionError("HTTPConnection(host='host.docker.internal', port=1337): Failed to resolve 'host.docker.internal' ([Errno -2] Name or service not known)"))

Generating:   2%|▏         | 4/200 [02:09<1:45:47, 32.39s/it]

  [attempt 1/6] local server error: HTTPConnectionPool(host='host.docker.internal', port=1337): Max retries exceeded with url: /v1/chat/completions (Caused by NameResolutionError("HTTPConnection(host='host.docker.internal', port=1337): Failed to resolve 'host.docker.internal' ([Errno -2] Name or service not known)")). Retrying in 2.4s ...
  [attempt 2/6] local server error: HTTPConnectionPool(host='host.docker.internal', port=1337): Max retries exceeded with url: /v1/chat/completions (Caused by NameResolutionError("HTTPConnection(host='host.docker.internal', port=1337): Failed to resolve 'host.docker.internal' ([Errno -2] Name or service not known)")). Retrying in 4.1s ...
  [attempt 3/6] local server error: HTTPConnectionPool(host='host.docker.internal', port=1337): Max retries exceeded with url: /v1/chat/completions (Caused by NameResolutionError("HTTPConnection(host='host.docker.internal', port=1337): Failed to resolve 'host.docker.internal' ([Errno -2] Name or service not known)"))

...

  [attempt 1/6] local server error: HTTPConnectionPool(host='host.docker.internal', port=1337): Max retries exceeded with url: /v1/chat/completions (Caused by NameResolutionError("HTTPConnection(host='host.docker.internal', port=1337): Failed to resolve 'host.docker.internal' ([Errno -2] Name or service not known)")). Retrying in 2.1s ...
  [attempt 2/6] local server error: HTTPConnectionPool(host='host.docker.internal', port=1337): Max retries exceeded with url: /v1/chat/completions (Caused by NameResolutionError("HTTPConnection(host='host.docker.internal', port=1337): Failed to resolve 'host.docker.internal' ([Errno -2] Name or service not known)")). Retrying in 4.2s ...
  [attempt 3/6] local server error: HTTPConnectionPool(host='host.docker.internal', port=1337): Max retries exceeded with url: /v1/chat/completions (Caused by NameResolutionError("HTTPConnection(host='host.docker.internal', port=1337): Failed to resolve 'host.docker.internal' ([Errno -2] Name or service not known)"))

Generating:   4%|▎         | 7/200 [08:40<4:57:37, 92.53s/it]

  [attempt 1/6] local server error: HTTPConnectionPool(host='host.docker.internal', port=1337): Max retries exceeded with url: /v1/chat/completions (Caused by NameResolutionError("HTTPConnection(host='host.docker.internal', port=1337): Failed to resolve 'host.docker.internal' ([Errno -2] Name or service not known)")). Retrying in 2.7s ...
  [attempt 2/6] local server error: HTTPConnectionPool(host='host.docker.internal', port=1337): Max retries exceeded with url: /v1/chat/completions (Caused by NameResolutionError("HTTPConnection(host='host.docker.internal', port=1337): Failed to resolve 'host.docker.internal' ([Errno -2] Name or service not known)")). Retrying in 4.2s ...
  [attempt 3/6] local server error: HTTPConnectionPool(host='host.docker.internal', port=1337): Max retries exceeded with url: /v1/chat/completions (Caused by NameResolutionError("HTTPConnection(host='host.docker.internal', port=1337): Failed to resolve 'host.docker.internal' ([Errno -2] Name or service not known)"))

Generating:   4%|▎         | 7/200 [09:06<4:11:04, 78.05s/it]


KeyboardInterrupt: 

In [ ]:
# Preview a few samples
with open(DATASET_PATH) as fh:
    lines = fh.readlines()
print(f"Total samples: {len(lines)}")
sample = json.loads(lines[0])
print("\n--- Instruction ---")
print(sample["instruction"])
print("\n--- Output (first 500 chars) ---")
print(sample["output"][:500])

## 2. Evaluate the Base Model (Before Fine-Tuning)

We run the student model on the eval test cases defined in `prompts.json` and
save each result as an HTML file. Open them in a browser to visually inspect quality.
The eval prompts are intentionally **different** from the training topics so we
measure genuine generalisation, not memorisation.

The cell imports `evaluate` directly from `evaluate_model.py` so model loading
progress and per-case generation output appears live in the cell.

In [ ]:
from evaluate_model import evaluate

base_eval_results = evaluate(
    model_name=STUDENT_MODEL,
    output_dir=EVAL_BASE_DIR,
    prompts_file=PROMPTS_FILE,
)

In [ ]:
# Display the base-model summary (uses the return value from the cell above)
for r in base_eval_results:
    status = "✓" if (r["starts_with_html"] and r["has_body"]) else "✗"
    print(f"{status} {r['id']:24s}  chars={r['char_count']:5d}")

In [ ]:
# Render the first eval result inline
from IPython.display import IFrame
first_eval_id = prompts['eval_test_cases'][0]['id']
IFrame(src=os.path.join(EVAL_BASE_DIR, f"{first_eval_id}.html"), width="100%", height=500)

## 3. Fine-Tune with Unsloth

We apply **QLoRA** (4-bit quantised LoRA) using Unsloth's optimised kernels.
Expected training time on an RTX 2080 Ti: ~10–20 minutes for 200 samples × 3 epochs.

The cell imports `train` directly from `train_model.py` so all training logs
(loss, learning rate, etc.) appear live in the cell.

In [ ]:
from train_model import train

train(
    model_name=STUDENT_MODEL,
    dataset_path=DATASET_PATH,
    output_dir=ADAPTER_DIR,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    grad_accum=GRAD_ACCUM,
)

## 3b. (Optional) Export Fine-Tuned Adapter to GGUF

Unsloth can merge the LoRA adapter into the base model and export the result
as a **GGUF** file, ready for local inference with **llama.cpp** or **Ollama**.

`save_gguf()` is a separate function in `train_model.py` that:
1. Validates the adapter directory and (if supplied) a local base-model path.
2. Loads the base model + LoRA adapter.
3. Calls Unsloth's `save_pretrained_gguf` with the chosen quantization.

Supported quantization methods: `q4_k_m` *(default — recommended)*, `q5_k_m`,
`q8_0`, `f16`, `q4_0`, `q5_0`, `q2_k`, `q3_k_m`.

In [ ]:
from train_model import save_gguf

gguf_path = save_gguf(
    adapter_dir=ADAPTER_DIR,
    output_dir=GGUF_DIR,
    quantization_method=GGUF_QUANT,
    model_name=STUDENT_MODEL,
)
print(f"GGUF export complete: {gguf_path}")

## 4. Evaluate the Fine-Tuned Model

Same eval test cases from `prompts.json`, now run with the LoRA adapter applied.

In [ ]:
from evaluate_model import evaluate

ft_eval_results = evaluate(
    model_name=STUDENT_MODEL,
    adapter_path=ADAPTER_DIR,
    output_dir=EVAL_FT_DIR,
    prompts_file=PROMPTS_FILE,
)

## 5. Before / After Comparison

In [ ]:
# Build lookup dicts from the results returned by the evaluate() calls above.
# If you restarted the kernel, load from the saved summary files instead:
#   with open(os.path.join(EVAL_BASE_DIR, "summary.json")) as fh:
#       base_eval_results = json.load(fh)
#   with open(os.path.join(EVAL_FT_DIR, "summary.json")) as fh:
#       ft_eval_results = json.load(fh)

base_by_id = {r["id"]: r for r in base_eval_results}
ft_by_id   = {r["id"]: r for r in ft_eval_results}

print(f"{'Test Case':<26} {'Base chars':>12} {'FT chars':>10} {'Base ✓':>8} {'FT ✓':>6}")
print("-" * 66)
for tc_id in base_by_id:
    b = base_by_id[tc_id]
    f = ft_by_id.get(tc_id, {})
    b_ok = "✓" if (b.get("starts_with_html") and b.get("has_body")) else "✗"
    f_ok = "✓" if (f.get("starts_with_html") and f.get("has_body")) else "✗"
    print(f"{tc_id:<26} {b['char_count']:>12} {f.get('char_count', 0):>10} {b_ok:>8} {f_ok:>6}")

In [ ]:
# Side-by-side comparison — change first_eval_id to any eval case id you like
from IPython.display import display, HTML

display(HTML(f"<h3>Base model — {first_eval_id}</h3>"))
display(IFrame(src=os.path.join(EVAL_BASE_DIR, f"{first_eval_id}.html"), width="100%", height=400))

display(HTML(f"<h3>Fine-tuned model — {first_eval_id}</h3>"))
display(IFrame(src=os.path.join(EVAL_FT_DIR, f"{first_eval_id}.html"), width="100%", height=400))

## 6. (Optional) Interactive Inference

In [ ]:
from evaluate_model import load_model_and_tokenizer, generate_html, extract_html, SYSTEM_PROMPT

inf_model, inf_tokenizer = load_model_and_tokenizer(STUDENT_MODEL, ADAPTER_DIR)
print("Model ready.")

In [ ]:
# Try your own prompt!
raw = generate_html(
    inf_model,
    inf_tokenizer,
    "Create a single-file HTML website: a dark-mode toggle demo page.",
    max_new_tokens=2048,
)
html = extract_html(raw)
print(html[:800])

In [ ]:
# Render inline
from IPython.display import display, HTML as _HTML
display(_HTML(html))

## 7. (Optional) Push Adapter to HuggingFace Hub

In [ ]:
# Uncomment and fill in your HF repo id to publish the adapter
# HF_REPO = "your-username/llama-3.2-3b-website-lora"
# from huggingface_hub import login
# login(token=HF_TOKEN)
# from train_model import train
# train(
#     model_name=STUDENT_MODEL,
#     dataset_path=DATASET_PATH,
#     output_dir=ADAPTER_DIR,
#     push_to_hub=HF_REPO,
# )
# print(f"Pushed to https://huggingface.co/{HF_REPO}")